# VLM basics: models that see

Two steps, two small models (both run on a laptop CPU):

1. **CLIP** ([openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32), ~600 MB) – puts images and text into the same embedding space. We use it to classify images into classes it was **never trained on** (zero-shot).
2. **SmolVLM** ([HuggingFaceTB/SmolVLM-256M-Instruct](https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct), ~500 MB) – a tiny vision-language model: give it an image and a question, it answers in text.

Remember the architecture from the class notes: *vision encoder → projector → LLM*. CLIP is the "vision encoder" idea on its own; SmolVLM is the whole thing.

## Setup

In [ ]:
%pip install -q transformers torch pillow requests

In [ ]:
import io
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image


def load_image(source):
    """Load an image from a URL or from a file on your disk."""
    if source.startswith("http"):
        response = requests.get(source, timeout=30)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content)).convert("RGB")
    return Image.open(source).convert("RGB")


# Images from the COCO dataset – or put the path to your own photo here
IMAGE_URLS = {
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "bear": "http://images.cocodataset.org/val2017/000000000285.jpg",
    "stop sign": "http://images.cocodataset.org/val2017/000000000724.jpg",
}
images = {name: load_image(url) for name, url in IMAGE_URLS.items()}

plt.figure(figsize=(12, 4))
for n, (name, img) in enumerate(images.items()):
    plt.subplot(1, len(images), n + 1)
    plt.imshow(img)
    plt.title(name)
    plt.axis("off")
plt.show()

## 1. CLIP – zero-shot classification

CLIP turns an image into a vector and a sentence into a vector **in the same space**. Matching pairs are close. So to classify an image we just write our classes as sentences – *"a photo of a cat"* – and pick the one whose vector is closest to the image's vector.

No training, no labelled data for our classes. In Class 4 we needed 2 000 labelled pictures to tell cats from dogs.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


@torch.no_grad()
def zero_shot(image, labels, template="a photo of a {}"):
    texts = [template.format(label) for label in labels]
    inputs = clip_processor(text=texts, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[0]     # similarity → probabilities
    for label, p in sorted(zip(labels, probs.tolist()), key=lambda x: -x[1]):
        print(f"   {label:<15} {p:.3f}")


labels = ["cat", "dog", "bear", "car", "stop sign", "remote control"]
for name, img in images.items():
    print(name)
    zero_shot(img, labels)

The labels are just text – change them to anything you like. CLIP even handles things like counts, colours or styles, though less reliably:

In [ ]:
zero_shot(images["cats"], ["one cat", "two cats", "three cats"], template="a photo of {}")
zero_shot(images["cats"], ["a pink sofa", "a green sofa", "a black sofa"], template="{}")

**The other direction: text → image search.** The same similarity scores let us find the best image for a description – this is how image search and many recommendation systems work.

In [ ]:
@torch.no_grad()
def search(query):
    names = list(images)
    inputs = clip_processor(text=[query], images=[images[n] for n in names], return_tensors="pt", padding=True)
    scores = clip_model(**inputs).logits_per_text.softmax(dim=1)[0]
    best = names[int(scores.argmax())]
    print(f"{query!r} → {best}  ({dict(zip(names, [round(s, 3) for s in scores.tolist()]))})")


search("an animal in the wild")
search("something you see on the street")
search("pets relaxing at home")

## 2. SmolVLM – ask questions about an image

Now the full vision-language model. The image is cut into patches, encoded, projected into "image tokens" and put in front of the text prompt. Then a (small) language model generates the answer token by token – just like in the LLM notebook.

Expect mistakes: this is one of the smallest VLMs that exist (256M parameters – the big ones have hundreds of billions). On a laptop CPU each answer takes from a few seconds up to a minute – the image alone is ~870 tokens.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

VLM = "HuggingFaceTB/SmolVLM-256M-Instruct"
vlm_processor = AutoProcessor.from_pretrained(VLM)
vlm_model = AutoModelForImageTextToText.from_pretrained(VLM)
vlm_model.eval()


@torch.no_grad()
def ask(image, question, max_new_tokens=80):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(text=prompt, images=[image], return_tensors="pt")
    out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = vlm_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return answer.strip()


print(ask(images["cats"], "Describe this image in one sentence."))

In [ ]:
for question in ["How many cats are there?", "What are the cats lying on?", "What objects are next to the cats?"]:
    print(f"Q: {question}\nA: {ask(images['cats'], question)}\n")

In [ ]:
print(ask(images["stop sign"], "What does the sign say, and where is it?"))
print(ask(images["bear"], "Is this animal dangerous? Answer in one sentence."))

Peek under the hood: how many tokens does the image take compared with the question?

In [ ]:
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "How many cats are there?"}]}]
prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = vlm_processor(text=prompt, images=[images["cats"]], return_tensors="pt")
n_question = len(vlm_processor.tokenizer("How many cats are there?")["input_ids"])
print(f"Whole prompt: {inputs['input_ids'].shape[1]} tokens, of which the question is only ~{n_question}")

Most of the prompt is the image. That's why pictures are "expensive" for VLMs, and why big models are careful about image resolution.

## Your turn

1. Use your own photo (`load_image("path/to/photo.jpg")`). Give CLIP labels that are *hard* to tell apart. When is it confidently wrong?
2. Change the CLIP template from `"a photo of a {}"` to just `"{}"`. Does accuracy change? (The CLIP authors found that prompt wording matters – "prompt engineering" for images.)
3. Ask SmolVLM a question whose answer is **not** in the image ("What's the name of the cat?"). Does it admit it doesn't know, or does it hallucinate?
4. Count objects: find an image with 5+ objects of one kind. Who counts better – CLIP or SmolVLM? Why might both struggle?